In [1]:
!pip install datasets bitsandbytes trl transformers peft huggingface-hub accelerate safetensors pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 202.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 201.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.7/781.7 kB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 157.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.2.0
    Uninstalling fsspec-2025.2.0:
      Successfully uninstalled fsspec-2025.2.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.9
    Uninstalling dill-0.3.9:
      Successfully uninstalled dill-0.3.9
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.17
    Uninstalling multiprocess-0.70.17:
      Successfully uninstalled multiprocess-0.70.17
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. 

In [2]:
!pip install --upgrade evaluate

In [3]:
#!pip install bitsandbytes

In [1]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
import torch
from torch.utils.data import Dataset
from tqdm import tqdm
import evaluate
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

from peft import get_peft_model, LoraConfig, TaskType, PeftModel

import pickle
import json
import matplotlib.pyplot as plt

from urllib.request import urlopen
import io

Matplotlib is building the font cache; this may take a moment.


In [2]:
import torch
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, DataCollatorForLanguageModeling, DataCollatorWithPadding, DataCollatorWithFlattening, BitsAndBytesConfig
from trl import setup_chat_format, DataCollatorForCompletionOnlyLM
from trl.extras.dataset_formatting import FORMAT_MAPPING, instructions_formatting_function, conversations_formatting_function
from trl.trainer import ConstantLengthDataset

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
device

device(type='cuda')

In [5]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/WzOT_CwDALWedTtXjwH7bA/CodeAlpaca-20k.json

--2025-04-17 05:57:46--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/WzOT_CwDALWedTtXjwH7bA/CodeAlpaca-20k.json
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6957007 (6.6M) [application/json]
Saving to: ‘CodeAlpaca-20k.json.4’

100%[======================================>] 6,957,007   9.64MB/s   in 0.7s   

2025-04-17 05:57:47 (9.64 MB/s) - ‘CodeAlpaca-20k.json.4’ saved [6957007/6957007]



In [6]:
dataset = load_dataset("json", data_files="CodeAlpaca-20k.json", split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['output', 'instruction', 'input'],
    num_rows: 20022
})

In [7]:
dataset[0]

{'output': 'arr = [2, 4, 6, 8, 10]',
 'instruction': 'Create an array of length 5 which contains all even numbers between 1 and 10.',
 'input': ''}

In [8]:
dataset = dataset.filter(lambda example: example["input"] == '')

Filter:   0%|          | 0/20022 [00:00<?, ? examples/s]

In [9]:
dataset = dataset.shuffle(seed=42)

In [10]:
dataset[0]

{'output': 'def create_input_string(nums, ops):\n    input_str = ""\n    \n    for i in range(len(nums)):\n        if i == 0:\n            input_str += str(nums[i])\n        else:\n            input_str += ops[i - 1] + str(nums[i])\n    \n    return input_str',
 'instruction': 'Create a function that produces input strings for a calculator.',
 'input': ''}

In [11]:
print(dataset[0]['output'])

def create_input_string(nums, ops):
    input_str = ""
    
    for i in range(len(nums)):
        if i == 0:
            input_str += str(nums[i])
        else:
            input_str += ops[i - 1] + str(nums[i])
    
    return input_str


In [12]:
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']
dataset_split

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 7811
    })
    test: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 1953
    })
})

In [13]:
# Select a small set of data for the resource limitation
# This dataset will be only used for evaluation parts, not for the training
#tiny_test_dataset=test_dataset.select(range(10))
#tiny_train_dataset=train_dataset.select(range(10))

In [14]:
# Base model
#model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m").to(device)

In [15]:
!pip install torchinfo
from torchinfo import summary

In [16]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m", padding_side='left')

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

In [17]:
tokenizer

GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [18]:
tokenizer.eos_token

'</s>'

In [19]:
print(tokenizer.chat_template)

None


In [31]:
tokenizer.special_tokens_map

{'bos_token': '</s>',
 'eos_token': '</s>',
 'unk_token': '</s>',
 'pad_token': '<pad>'}

In [32]:
def get_multiple_of(vocab_size):
    return 2**(bin(vocab_size)[::-1].find('1'))

##pad_to_multiple_of = get_multiple_of(model.config.vocab_size)
#pad_to_multiple_of

In [33]:
#model.resize_token_embeddings(len(tokenizer),
#                                  pad_to_multiple_of=pad_to_multiple_of)

In [34]:
tokenizer_phi = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-4k-instruct")
print(tokenizer_phi.chat_template)

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [35]:
tokenizer_phi

LlamaTokenizerFast(name_or_path='microsoft/phi-3-mini-4k-instruct', vocab_size=32000, model_max_length=4096, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=False),
	32000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<|placeholder1|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=Tr

In [36]:
tokenizer

GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [37]:
tokenizer.special_tokens_map

{'bos_token': '</s>',
 'eos_token': '</s>',
 'unk_token': '</s>',
 'pad_token': '<pad>'}

In [38]:
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'pad_token': '<|endoftext|>'}

In [39]:
tokenizer.vocab['</s>']

2

In [40]:
tokenizer_phi.vocab["<|user|>"]

32010

In [41]:
tokenizer.vocab['<pad>']

1

In [42]:
special_tokens_dict = {'unk_token': '<unk>',
                       'eos_token': '<|endoftext|>',
                       }
special_tokens_dict

{'unk_token': '<unk>', 'eos_token': '<|endoftext|>'}

In [43]:
new_toks  = ['<|system|>', '<|user|>', '<|assistant|>', '<|end|>']

In [44]:
new_toks

['<|system|>', '<|user|>', '<|assistant|>', '<|end|>']

In [45]:
tokenizer.add_special_tokens(special_tokens_dict)

2

In [46]:
tokenizer

GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50260: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [47]:
tokenizer.add_tokens(new_toks)

4

In [48]:
tokenizer

GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50260: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50265: AddedToken("<|system|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50266: AddedToken("<|user|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),


In [49]:
len(tokenizer)

50269

In [50]:
tokenizer.chat_template = tokenizer_phi.chat_template
print(tokenizer.chat_template)

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [51]:
def ds_chat_format(example):
  messages = []
  user_dict = {}
  asst_dict = {}
  for k in example.keys():
    role = k
    content = example[k]
    if k == 'assistant':
      asst_dict = {'role': role, 'content': content}
    else:
      user_dict = {'role': role, 'content': content}
  messages = [user_dict, asst_dict]
  return {'messages': messages}

In [52]:
import os
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

In [53]:
bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.float32
)
repo_id = "facebook/opt-350m"
model = AutoModelForCausalLM.from_pretrained(repo_id,
                                             device_map="cuda:0",
                                             quantization_config=bnb_config
)

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [54]:
summary(model)

Layer (type:depth-idx)                             Param #
OPTForCausalLM                                     --
├─OPTModel: 1-1                                    --
│    └─OPTDecoder: 2-1                             --
│    │    └─Embedding: 3-1                         25,739,264
│    │    └─OPTLearnedPositionalEmbedding: 3-2     2,099,200
│    │    └─Linear4bit: 3-3                        (262,144)
│    │    └─Linear4bit: 3-4                        (262,144)
│    │    └─ModuleList: 3-5                        151,314,432
├─Linear: 1-2                                      25,739,264
Total params: 205,416,448
Trainable params: 53,676,032
Non-trainable params: 151,740,416

In [55]:
print(model.get_memory_footprint()/1e6)

207.835136


In [56]:
model = prepare_model_for_kbit_training(model)
print(summary(model))
config = LoraConfig(
    r=8,                   # the rank of the adapter, the lower the fewer parameters you'll need to train
    lora_alpha=16,         # multiplier, usually 2*r
    bias="none",           # BEWARE: training biases *modifies* base model's behavior
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    # Newer models, such as Phi-3 at time of writing, may require
    # manually setting target modules
    #target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj'],
)

model = get_peft_model(model, config)
model

Layer (type:depth-idx)                             Param #
OPTForCausalLM                                     --
├─OPTModel: 1-1                                    --
│    └─OPTDecoder: 2-1                             --
│    │    └─Embedding: 3-1                         (25,739,264)
│    │    └─OPTLearnedPositionalEmbedding: 3-2     (2,099,200)
│    │    └─Linear4bit: 3-3                        (262,144)
│    │    └─Linear4bit: 3-4                        (262,144)
│    │    └─ModuleList: 3-5                        (151,314,432)
├─Linear: 1-2                                      (25,739,264)
Total params: 205,416,448
Trainable params: 0
Non-trainable params: 205,416,448


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear4bit(in_features=1024, out_features=512, bias=False)
          (project_in): Linear4bit(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTSdpaAttention(
                (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out

In [57]:
print(model.get_memory_footprint()/1e6)

267.296768


In [58]:
trainable_parms, tot_parms = model.get_nb_trainable_parameters()
print(f'Trainable parameters:             {trainable_parms/1e6:.2f}M')
print(f'Total parameters:                 {tot_parms/1e6:.2f}M')
print(f'Fraction of trainable parameters: {100*trainable_parms/tot_parms:.2f}%')

Trainable parameters:             0.79M
Total parameters:                 331.98M
Fraction of trainable parameters: 0.24%


In [59]:
train_dataset

Dataset({
    features: ['output', 'instruction', 'input'],
    num_rows: 7811
})

In [60]:
test_dataset

Dataset({
    features: ['output', 'instruction', 'input'],
    num_rows: 1953
})

In [61]:
train_dataset = train_dataset.rename_column("instruction", "user")
train_dataset = train_dataset.rename_column("output", "assistant")
train_dataset

Dataset({
    features: ['assistant', 'user', 'input'],
    num_rows: 7811
})

In [62]:
train_dataset = train_dataset.remove_columns("input")
train_dataset

Dataset({
    features: ['assistant', 'user'],
    num_rows: 7811
})

In [63]:
test_dataset = test_dataset.rename_column("instruction", "user")
test_dataset = test_dataset.rename_column("output", "assistant")
test_dataset = test_dataset.remove_columns("input")
test_dataset

Dataset({
    features: ['assistant', 'user'],
    num_rows: 1953
})

In [64]:
train_dataset[0]

{'assistant': 'db.collection.find( { count: { $gt: 10 } } )',
 'user': 'Write a query in MongoDB to find all documents which have a count greater than 10.'}

In [65]:
train_ds = train_dataset.map(ds_chat_format, batched=False)
test_ds = test_dataset.map(ds_chat_format, batched=False)

Map:   0%|          | 0/7811 [00:00<?, ? examples/s]

Map:   0%|          | 0/1953 [00:00<?, ? examples/s]

In [66]:
train_ds['messages'][0:2]

[[{'content': 'Write a query in MongoDB to find all documents which have a count greater than 10.',
   'role': 'user'},
  {'content': 'db.collection.find( { count: { $gt: 10 } } )',
   'role': 'assistant'}],
 [{'content': 'Determine the fraction of numbers in this list that are multiples of 3: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10].',
   'role': 'user'},
  {'content': 'fraction = 3/10 = 0.3', 'role': 'assistant'}]]

In [67]:
test_ds['messages'][0:2]

[[{'content': 'Name the most important benefit of using a database system.',
   'role': 'user'},
  {'content': 'The most important benefit of using a database system is the ability to store and retrieve data quickly and easily. Database systems also provide support for data security, data integrity, and concurrently accessing and modifying data from multiple systems.',
   'role': 'assistant'}],
 [{'content': 'Come up with a Java program that checks if one string is a substring of another.',
   'role': 'user'},
  {'content': 'public static boolean isSubstring(String s, String x) {\n    int i = 0, j = 0;\n    while (i < s.length() && j < x.length()) {\n        if (s.charAt(i) == x.charAt(j)) {\n            i++;\n            j++;\n        } else {\n            i = i - j + 1;\n            j = 0;\n        }\n    }\n    if (j == x.length()) {\n        return true;\n    }\n    return false;\n}',
   'role': 'assistant'}]]

In [68]:
FORMAT_MAPPING['chatml'] == train_ds.features['messages']

True

In [69]:
messages = train_ds["messages"][0]
output_texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(output_texts)

<|user|>
Write a query in MongoDB to find all documents which have a count greater than 10.<|end|>
<|assistant|>
db.collection.find( { count: { $gt: 10 } } )<|end|>
<|endoftext|>


In [70]:
train_ds, test_ds

(Dataset({
     features: ['assistant', 'user', 'messages'],
     num_rows: 7811
 }),
 Dataset({
     features: ['assistant', 'user', 'messages'],
     num_rows: 1953
 }))

In [71]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage
    # These arguments will squeeze the most out of your GPU's RAM
    # Checkpointing
    #gradient_checkpointing=True,
    # this saves a LOT of memory
    # Set this to avoid exceptions in newer versions of PyTorch
    #gradient_checkpointing_kwargs={'use_reentrant': False},
    # Gradient Accumulation / Batch size
    # Actual batch (for updating) is same (1x) as micro-batch size
    gradient_accumulation_steps=1,
    # The initial (micro) batch size to start off with
    per_device_train_batch_size=64,
    # If batch size would cause OOM, halves its size until it works
    auto_find_batch_size=True,

    ## GROUP 2: Dataset-related
    max_seq_length=256,
    # Dataset
    # packing a dataset means no padding is needed
    packing=True,

    ## GROUP 3: These are typical training parameters
    num_train_epochs=20,
    learning_rate=3e-4,
    # Optimizer
    # 8-bit Adam optimizer - doesn't help much if you're using LoRA!
    optim='adamw_torch',

    ## GROUP 4: Logging parameters
    logging_steps=50,
    logging_dir='./logs',
    output_dir='./opt350-alpaca-code-adapter',
    report_to='none'
)

In [72]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_ds
    )

Converting train dataset to ChatML:   0%|          | 0/7811 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/7811 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7811 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/7811 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [73]:
trainer.__dict__

{'_metrics': {'train': defaultdict(list, {}), 'eval': defaultdict(list, {})},
 '_total_train_tokens': 0,
 'args': SFTConfig(output_dir='./opt350-alpaca-code-adapter', overwrite_output_dir=False, do_train=False, do_eval=False, do_predict=False, eval_strategy=<IntervalStrategy.NO: 'no'>, prediction_loss_only=False, per_device_train_batch_size=64, per_device_eval_batch_size=8, per_gpu_train_batch_size=None, per_gpu_eval_batch_size=None, gradient_accumulation_steps=1, eval_accumulation_steps=None, eval_delay=0, torch_empty_cache_steps=None, learning_rate=0.0003, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=20, max_steps=-1, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs={}, warmup_ratio=0.0, warmup_steps=0, log_level='passive', log_level_replica='warning', log_on_each_node=True, logging_dir='./logs', logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_first_step=False, logging_steps=50, logging_

In [74]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))

In [75]:
tokenizer

GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50260: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50265: AddedToken("<|system|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50266: AddedToken("<|user|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),


In [76]:
batch['input_ids'][0], batch['labels'][0]

(tensor([  311,    10,  5059,   618,    19,     5,  1270,     6,    22,  2387,
          1234, 21617,  1869,    72, 50268, 50118, 50267, 50118, 41552,   328,
         19174,  7164,   975, 16035, 48445, 15698, 50118, 41552,  6660, 22682,
         40635,   225, 46479, 50118, 41552,  3628, 15698, 50118, 28696, 46876,
         49160,   594, 40635, 44987,    12,   398, 46479, 50118, 28696, 46876,
           766, 40635,  5877,  3427,   113,  1383, 40635, 36097,  5214, 42005,
            12, 36097,     6,  2557,    12,  8056,  5214,   134,     4,   288,
         46479, 50118, 28696, 14691, 15698,  2387,  1234, 21617,  1869, 49138,
         14691, 15698, 50118, 49138,  3628, 15698, 50118, 41552,  9773, 15698,
         50118, 28696,   298,   134, 15698,  2387,  1234, 21617,  1869, 49138,
           298,   134, 15698, 50118, 28696,   642, 15698, 48386,  5059,   618,
          1383,   259, 49803,   642, 15698, 50118, 49138,  9773, 15698, 50118,
         49138,  6660, 15698, 50268, 50118, 50260,  

In [77]:
#tokenizer.decode(batch['labels'][0][batch['labels'][0]>-100])

In [78]:
#tokenizer.decode(batch['input_ids'][0][batch['input_ids'][0] != 1])

In [79]:
len(batch['input_ids'][0])

256

In [80]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
50,2.270700
100,1.897400
150,1.805100
200,1.753200
250,1.716500
300,1.699900
350,1.663200
400,1.649700
450,1.630500
500,1.607800


TrainOutput(global_step=1060, training_loss=1.6606299274372605, metrics={'train_runtime': 9008.9202, 'train_samples_per_second': 7.457, 'train_steps_per_second': 0.118, 'total_flos': 3.138419668549632e+16, 'train_loss': 1.6606299274372605})

In [81]:
def gen_prompt(tokenizer, sentence):
    converted_sample = [
        {"role": "user", "content": sentence},
    ]
    prompt = tokenizer.apply_chat_template(converted_sample,
                                           tokenize=False,
                                           add_generation_prompt=True)
    return prompt

In [110]:
def generate(model, tokenizer, prompt, orig_completion, max_new_tokens=128, skip_special_tokens=False):
    tokenized_input = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)

    model.eval()
    generation_output = model.generate(**tokenized_input,
                                       eos_token_id=tokenizer.eos_token_id,
                                       max_new_tokens=max_new_tokens)

    output = tokenizer.batch_decode(generation_output,
                                    skip_special_tokens=skip_special_tokens)
    print(output[0])
    print("##############")
    print('orig completion: ', orig_completion)
    #return output[0]

In [114]:
def gen_prompt_exec(model, tokenizer, sentence, orig_completion):
    prompt = gen_prompt(tokenizer, sentence)
    print("##############")
    print(prompt)
    print("##############")
    generate(model, tokenizer, prompt, orig_completion)

In [117]:
sentence = test_dataset[0]['user']
orig_completion = test_dataset[0]['assistant']
gen_prompt_exec(model, tokenizer, sentence, orig_completion)

##############
<|user|>
Name the most important benefit of using a database system.<|end|>
<|assistant|>

##############
<|user|>
Name the most important benefit of using a database system.<|end|>
<|assistant|>
The most important benefit of using a database system is the ability to store and retrieve data in a way that is easy to use and maintain. This is especially true for data that is stored in a database, such as financial data, customer data, and customer information. This is especially true for data that is stored in a database, such as customer data, customer information, and customer information.<|end|>
<|endoftext|>
##############
orig completion:  The most important benefit of using a database system is the ability to store and retrieve data quickly and easily. Database systems also provide support for data security, data integrity, and concurrently accessing and modifying data from multiple systems.


In [118]:
idx = 6
sentence = test_dataset[idx]['user']
orig_completion = test_dataset[idx]['assistant']
gen_prompt_exec(model, tokenizer, sentence, orig_completion)

##############
<|user|>
What is an example of a multi-dimensional array?<|end|>
<|assistant|>

##############
<|user|>
What is an example of a multi-dimensional array?<|end|>
<|assistant|>
A multi-dimensional array is a set of objects that are arranged in a way that is more or less parallel to the original object. It is a set of objects that are arranged in a way that is more or less parallel to the original object. It is a set of objects that are arranged in a way that is more or less parallel to the original object. It is a set of objects that are arranged in a way that is more or less parallel to the original object.<|end|>
<|endoftext|>
##############
orig completion:  An example of a multi-dimensional array is a two-dimensional array, which is an array of arrays. For example:

var array = [[1,2,3], [4,5,6], [7,8,9]];


In [126]:
idx = 77
sentence = train_dataset[idx]['user']
orig_completion = train_dataset[idx]['assistant']
gen_prompt_exec(model, tokenizer, sentence, orig_completion)

##############
<|user|>
Construct a while loop with an if-else statement in Java to print all odd numbers between 1 to 10.<|end|>
<|assistant|>

##############
<|user|>
Construct a while loop with an if-else statement in Java to print all odd numbers between 1 to 10.<|end|>
<|assistant|>
public static int getUnodds(int nums) {
    int num = num;
    for (int i = 0; i < num; i++) {
        int num = num;
        if (num % i == 0) {
             num += i;
        }
    }
    return num;
}<|end|>
<|endoftext|>
##############
orig completion:  int i = 1;
while (i <= 10) { 
    if (i % 2 != 0) {
        System.out.print(i + " "); 
    }
    i++;
}


In [127]:
trainer.save_model('alpaca-opt-code-adapter_full_text')

In [128]:
#let us continue training the same model but using completiononlyLM collator

In [143]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage
    # These arguments will squeeze the most out of your GPU's RAM
    # Checkpointing
    #gradient_checkpointing=True,
    # this saves a LOT of memory
    # Set this to avoid exceptions in newer versions of PyTorch
    #gradient_checkpointing_kwargs={'use_reentrant': False},
    # Gradient Accumulation / Batch size
    # Actual batch (for updating) is same (1x) as micro-batch size
    gradient_accumulation_steps=1,
    # The initial (micro) batch size to start off with
    per_device_train_batch_size=64,
    # If batch size would cause OOM, halves its size until it works
    auto_find_batch_size=True,

    ## GROUP 2: Dataset-related
    max_seq_length=128,
    # Dataset
    # packing a dataset means no padding is needed
    packing=False,

    ## GROUP 3: These are typical training parameters
    num_train_epochs=20,
    learning_rate=3e-4,
    # Optimizer
    # 8-bit Adam optimizer - doesn't help much if you're using LoRA!
    optim='adamw_torch',

    ## GROUP 4: Logging parameters
    logging_steps=50,
    logging_dir='./logs',
    output_dir='./opt350-alpaca-code-adapter',
    report_to='none'
)

In [144]:
print(tokenizer.chat_template)

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [145]:
train_dataset[0]

{'assistant': 'db.collection.find( { count: { $gt: 10 } } )',
 'user': 'Write a query in MongoDB to find all documents which have a count greater than 10.'}

In [146]:
tokenizer

GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50260: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50265: AddedToken("<|system|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50266: AddedToken("<|user|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),


In [147]:
tokenizer.padding_side='left'
response_template = '<|assistant|>' # according to the tokenizer's chat template
collator_fn=DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_ds,
    data_collator=collator_fn
    )

Applying chat template to train dataset:   0%|          | 0/7811 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7811 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7811 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    data_collator=collator_fn,
    args=SFTConfig(
        output_dir="./future_name_on_the_hub",
        packing=False,
        max_seq_length=max_seq_length,
    )
)

In [148]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))
batch['input_ids'][0], batch['labels'][0]

(tensor([    1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     2, 50266, 50118, 45714,    10, 41614,
         25860,     7,  5163,    70,     5, 11693,  1947,    11,     5,   576,
         41616,     4, 50268, 50118, 50267, 50118, 49179,   211, 11595,  2444,
          7164,   412, 11974, 41616,   131, 50268, 5

In [149]:
len(batch['input_ids'][0])

128

In [ ]:
trainer.train()

Step,Training Loss
50,1.448900
100,1.489700
150,1.419300
200,1.423900
250,1.414500
300,1.383800
350,1.390000
400,1.382700
450,1.356500
500,1.385900


TrainOutput(global_step=2460, training_loss=1.2849677585973973, metrics={'train_runtime': 10075.6906, 'train_samples_per_second': 15.505, 'train_steps_per_second': 0.244, 'total_flos': 3.648927294500045e+16, 'train_loss': 1.2849677585973973})

In [159]:
idx = 10
sentence = test_dataset[idx]['user']
orig_completion = test_dataset[idx]['assistant']
gen_prompt_exec(model, tokenizer, sentence, orig_completion)

##############
<|user|>
Write a code snipplet that computes the sum of numbers between 1 and 100.<|end|>
<|assistant|>

##############
<|user|>
Write a code snipplet that computes the sum of numbers between 1 and 100.<|end|>
<|assistant|>
sum_of_numbers = 0
for num in range(1, 100):
    sum_of_numbers += num
print(sum_of_numbers)<|end|>
<|endoftext|>
##############
orig completion:  let sum = 0;
for(let i = 1; i <= 100; i++) {
  sum += i;
}


In [160]:
trainer.save_model('alpaca-opt-code-adapter_full_text_CO')

In [163]:
trainer.args.num_train_epochs = 10

In [164]:
trainer.args.num_train_epochs

10

In [ ]:
trainer.train()

Step,Training Loss
50,1.199800
100,1.259100
150,1.210400
200,1.216500
250,1.221500
300,1.197200
350,1.206900
400,1.204900
450,1.188900
500,1.220300


TrainOutput(global_step=1230, training_loss=1.1831197630099164, metrics={'train_runtime': 5037.7143, 'train_samples_per_second': 15.505, 'train_steps_per_second': 0.244, 'total_flos': 1.824474322717901e+16, 'train_loss': 1.1831197630099164})

In [166]:
idx = 10
sentence = test_dataset[idx]['user']
orig_completion = test_dataset[idx]['assistant']
gen_prompt_exec(model, tokenizer, sentence, orig_completion)

##############
<|user|>
Write a code snipplet that computes the sum of numbers between 1 and 100.<|end|>
<|assistant|>

##############
<|user|>
Write a code snipplet that computes the sum of numbers between 1 and 100.<|end|>
<|assistant|>
sum = 0
for i in range(1, 101):
    sum += i
print(sum)<|end|>
<|endoftext|>
##############
orig completion:  let sum = 0;
for(let i = 1; i <= 100; i++) {
  sum += i;
}


In [168]:
idx = 25
sentence = test_dataset[idx]['user']
orig_completion = test_dataset[idx]['assistant']
gen_prompt_exec(model, tokenizer, sentence, orig_completion)

##############
<|user|>
Create a JavaScript object that contains a student's name, age, and courses.<|end|>
<|assistant|>

##############
<|user|>
Create a JavaScript object that contains a student's name, age, and courses.<|end|>
<|assistant|>
const studentName = {
    name: 'John',
    age: 25,
    courses: 'Degree',
    course_id: 'Degree',
    course_description: 'Degree'
};<|end|>
<|endoftext|>
##############
orig completion:  let student = { 
    name: "John Doe", 
    age: 20, 
    courses: ["Math", "Physics", "English"] 
};


In [169]:
idx = 5
sentence = test_dataset[idx]['user']
orig_completion = test_dataset[idx]['assistant']
gen_prompt_exec(model, tokenizer, sentence, orig_completion)

##############
<|user|>
Generate a 5x5 array with all its elements equal to 1.<|end|>
<|assistant|>

##############
<|user|>
Generate a 5x5 array with all its elements equal to 1.<|end|>
<|assistant|>
import random

def generate_array():
    array = []
    for i in range(5):
        array.append(random.randint(1, 1))
    return array<|end|>
<|endoftext|>
##############
orig completion:  arr = [[1,1,1,1,1],
       [1,1,1,1,1],
       [1,1,1,1,1],
       [1,1,1,1,1],
       [1,1,1,1,1]]


In [170]:
idx = 30
sentence = test_dataset[idx]['user']
orig_completion = test_dataset[idx]['assistant']
gen_prompt_exec(model, tokenizer, sentence, orig_completion)

##############
<|user|>
Construct a query in MySQL to calculate the total salary of all employees in a given department.<|end|>
<|assistant|>

##############
<|user|>
Construct a query in MySQL to calculate the total salary of all employees in a given department.<|end|>
<|assistant|>
SELECT SUM(salary) FROM employees WHERE department = 'Department1';<|end|>
<|endoftext|>
##############
orig completion:  SELECT SUM(Salary) 
FROM Employees 
WHERE Department = 'DEPARTMENT-NAME';


In [171]:
trainer.save_model('alpaca-opt-code-adapter_full_text_CO_1')

In [172]:
os.listdir('alpaca-opt-code-adapter_full_text_CO_1')

['vocab.json',
 'adapter_config.json',
 'tokenizer.json',
 'tokenizer_config.json',
 'added_tokens.json',
 'adapter_model.safetensors',
 'special_tokens_map.json',
 'training_args.bin',
 'README.md',
 'merges.txt']

In [173]:
from huggingface_hub import login
login()

In [174]:
trainer.push_to_hub()

adapter_model.safetensors:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.56k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Fardan/opt350-alpaca-code-adapter/commit/823a3c692e1d93a2b9deb3ed898f77546febbfae', commit_message='End of training', commit_description='', oid='823a3c692e1d93a2b9deb3ed898f77546febbfae', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Fardan/opt350-alpaca-code-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='Fardan/opt350-alpaca-code-adapter'), pr_revision=None, pr_num=None)